In [1]:
import pandas as pd


In [4]:
df = pd.read_csv(r"C:\Users\Admin\Documents\shopping_trends.csv")

In [5]:
df.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Payment Method,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Preferred Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Credit Card,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Bank Transfer,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Cash,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,PayPal,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Cash,Free Shipping,Yes,Yes,31,PayPal,Annually


In [8]:

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 19 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Customer ID               3900 non-null   int64  
 1   Age                       3900 non-null   int64  
 2   Gender                    3900 non-null   object 
 3   Item Purchased            3900 non-null   object 
 4   Category                  3900 non-null   object 
 5   Purchase Amount (USD)     3900 non-null   int64  
 6   Location                  3900 non-null   object 
 7   Size                      3900 non-null   object 
 8   Color                     3900 non-null   object 
 9   Season                    3900 non-null   object 
 10  Review Rating             3900 non-null   float64
 11  Subscription Status       3900 non-null   object 
 12  Payment Method            3900 non-null   object 
 13  Shipping Type             3900 non-null   object 
 14  Discount

In [19]:
#DATA CLEANING

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
)
print("\nCleaned column names:")
print(df.columns.tolist())
 
# ---------------------------------------------------------
# 3. CHECK & HANDLE MISSING VALUES
# ---------------------------------------------------------
print("\nMissing values per column:")
print(df.isnull().sum())
 
# Numeric columns: fill missing with median (robust to outliers)
numeric_cols = ["age", "purchase_amount_usd", "review_rating", "previous_purchases"]
for col in numeric_cols:
    if col in df.columns and df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())
 
# Categorical columns: fill missing with mode (most frequent value)
categorical_cols = [
    "gender", "item_purchased", "category", "location", "size", "color",
    "season", "subscription_status", "payment_method", "shipping_type",
    "discount_applied", "promo_code_used", "preferred_payment_method",
    "frequency_of_purchases"
]
for col in categorical_cols:
    if col in df.columns and df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])
 
# ---------------------------------------------------------
# 4. REMOVE DUPLICATES
# ---------------------------------------------------------
before = len(df)
df.drop_duplicates(inplace=True)
print(f"\nRemoved {before - len(df)} duplicate rows")
 
# ---------------------------------------------------------
# 5. FIX DATA TYPES
# ---------------------------------------------------------
df["age"] = df["age"].astype(int)
df["purchase_amount_usd"] = df["purchase_amount_usd"].astype(float)
df["review_rating"] = df["review_rating"].astype(float)
df["previous_purchases"] = df["previous_purchases"].astype(int)
 
# ---------------------------------------------------------
# 6. STANDARDIZE TEXT / CATEGORICAL VALUES
#    (strip whitespace, consistent casing — avoids "Yes"/"yes"/"YES"
#     being treated as different categories)
# ---------------------------------------------------------
text_cols = df.select_dtypes(include="object").columns
for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.title()
 
# Yes/No columns should be consistent (Title case turns them into "Yes"/"No")
yes_no_cols = ["subscription_status", "discount_applied", "promo_code_used"]
for col in yes_no_cols:
    if col in df.columns:
        df[col] = df[col].map({"Yes": "Yes", "No": "No"}).fillna(df[col])
 
# ---------------------------------------------------------
# 7. HANDLE OUTLIERS (IQR method) — applied to purchase_amount_usd
#    We cap rather than drop, so we don't lose rows.
# ---------------------------------------------------------
Q1 = df["purchase_amount_usd"].quantile(0.25)
Q3 = df["purchase_amount_usd"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
 
outliers = df[(df["purchase_amount_usd"] < lower_bound) | (df["purchase_amount_usd"] > upper_bound)]
print(f"\nOutliers detected in purchase_amount_usd: {len(outliers)}")
 
df["purchase_amount_usd"] = df["purchase_amount_usd"].clip(lower=lower_bound, upper=upper_bound)
 
# ---------------------------------------------------------
# 8. FEATURE ENGINEERING (new columns that add business value)
# ---------------------------------------------------------
 
# Age group buckets — useful for demographic segmentation
df["age_group"] = pd.cut(
    df["age"],
    bins=[0, 25, 35, 45, 55, 100],
    labels=["18-25", "26-35", "36-45", "46-55", "56+"]
)
 
# Binary flags (easier for BI tools & ML models than "Yes"/"No" strings)
df["is_subscribed"] = df["subscription_status"].map({"Yes": 1, "No": 0})
df["discount_used"] = df["discount_applied"].map({"Yes": 1, "No": 0})
df["promo_used"] = df["promo_code_used"].map({"Yes": 1, "No": 0})
 
# Spend tier — quick segmentation for dashboards
df["spend_tier"] = pd.cut(
    df["purchase_amount_usd"],
    bins=[0, 30, 60, 90, 200],
    labels=["Low", "Medium", "High", "Premium"]
)
 
# Customer value score (simple composite metric — spend x loyalty)
df["customer_value_score"] = (
    df["purchase_amount_usd"] * 0.5
    + df["previous_purchases"] * 2
    + df["review_rating"] * 5
).round(2)


Cleaned column names:
['customer_id', 'age', 'gender', 'item_purchased', 'category', 'purchase_amount_usd', 'location', 'size', 'color', 'season', 'review_rating', 'subscription_status', 'payment_method', 'shipping_type', 'discount_applied', 'promo_code_used', 'previous_purchases', 'preferred_payment_method', 'frequency_of_purchases', 'age_group', 'is_subscribed', 'discount_used', 'promo_used', 'spend_tier', 'customer_value_score']

Missing values per column:
customer_id                 0
age                         0
gender                      0
item_purchased              0
category                    0
purchase_amount_usd         0
location                    0
size                        0
color                       0
season                      0
review_rating               0
subscription_status         0
payment_method              0
shipping_type               0
discount_applied            0
promo_code_used             0
previous_purchases          0
preferred_payment_method 

In [15]:
df.head()

,customer_id,age,gender,item_purchased,category,purchase_amount_usd,location,size,color,season,...,promo_code_used,previous_purchases,preferred_payment_method,frequency_of_purchases,age_group,is_subscribed,discount_used,promo_used,spend_tier,customer_value_score
0,1,55,Male,Blouse,Clothing,53.0,Kentucky,L,Gray,Winter,...,Yes,14,Venmo,Fortnightly,46-55,1,1,1,Medium,70.0
1,2,19,Male,Sweater,Clothing,64.0,Maine,L,Maroon,Winter,...,Yes,2,Cash,Fortnightly,18-25,1,1,1,High,51.5
2,3,50,Male,Jeans,Clothing,73.0,Massachusetts,S,Maroon,Spring,...,Yes,23,Credit Card,Weekly,46-55,1,1,1,High,98.0
3,4,21,Male,Sandals,Footwear,90.0,Rhode Island,M,Maroon,Spring,...,Yes,49,Paypal,Weekly,18-25,1,1,1,High,160.5
4,5,45,Male,Blouse,Clothing,49.0,Oregon,M,Turquoise,Spring,...,Yes,31,Paypal,Annually,36-45,1,1,1,Medium,100.0


In [18]:
#FEATURE SELECTION TO REMOVE REDUNDANT COLUMN 

columns_to_drop = ["item_purchased"]  # too granular; category captures this better
df_selected = df.drop(columns=[c for c in columns_to_drop if c in df.columns])
 
# Reorder so engineered features sit near their source columns
final_columns = [
    "customer_id", "age", "age_group", "gender", "category",
    "purchase_amount_usd", "spend_tier", "location", "size", "color", "season",
    "review_rating", "subscription_status", "is_subscribed",
    "payment_method", "preferred_payment_method", "shipping_type",
    "discount_applied", "discount_used", "promo_code_used", "promo_used",
    "previous_purchases", "customer_value_score", "frequency_of_purchases"
]
final_columns = [c for c in final_columns if c in df_selected.columns]
df_final = df_selected[final_columns]

In [13]:
df_final.head()


,customer_id,age,age_group,gender,category,purchase_amount_usd,spend_tier,location,size,color,...,payment_method,preferred_payment_method,shipping_type,discount_applied,discount_used,promo_code_used,promo_used,previous_purchases,customer_value_score,frequency_of_purchases
0,1,55,46-55,Male,Clothing,53.0,Medium,Kentucky,L,Gray,...,Credit Card,Venmo,Express,Yes,1,Yes,1,14,70.0,Fortnightly
1,2,19,18-25,Male,Clothing,64.0,High,Maine,L,Maroon,...,Bank Transfer,Cash,Express,Yes,1,Yes,1,2,51.5,Fortnightly
2,3,50,46-55,Male,Clothing,73.0,High,Massachusetts,S,Maroon,...,Cash,Credit Card,Free Shipping,Yes,1,Yes,1,23,98.0,Weekly
3,4,21,18-25,Male,Footwear,90.0,High,Rhode Island,M,Maroon,...,Paypal,Paypal,Next Day Air,Yes,1,Yes,1,49,160.5,Weekly
4,5,45,36-45,Male,Clothing,49.0,Medium,Oregon,M,Turquoise,...,Cash,Paypal,Free Shipping,Yes,1,Yes,1,31,100.0,Annually


In [16]:
#ROUNDING OFF NUMERIC DATA

df_final["purchase_amount_usd"] = df_final["purchase_amount_usd"].round(2)
df_final["customer_value_score"] = df_final["customer_value_score"].round(2)
df_final["review_rating"] = df_final["review_rating"].round(1)

In [17]:
df_final.head()

,customer_id,age,age_group,gender,category,purchase_amount_usd,spend_tier,location,size,color,...,payment_method,preferred_payment_method,shipping_type,discount_applied,discount_used,promo_code_used,promo_used,previous_purchases,customer_value_score,frequency_of_purchases
0,1,55,46-55,Male,Clothing,53.0,Medium,Kentucky,L,Gray,...,Credit Card,Venmo,Express,Yes,1,Yes,1,14,70.0,Fortnightly
1,2,19,18-25,Male,Clothing,64.0,High,Maine,L,Maroon,...,Bank Transfer,Cash,Express,Yes,1,Yes,1,2,51.5,Fortnightly
2,3,50,46-55,Male,Clothing,73.0,High,Massachusetts,S,Maroon,...,Cash,Credit Card,Free Shipping,Yes,1,Yes,1,23,98.0,Weekly
3,4,21,18-25,Male,Footwear,90.0,High,Rhode Island,M,Maroon,...,Paypal,Paypal,Next Day Air,Yes,1,Yes,1,49,160.5,Weekly
4,5,45,36-45,Male,Clothing,49.0,Medium,Oregon,M,Turquoise,...,Cash,Paypal,Free Shipping,Yes,1,Yes,1,31,100.0,Annually


In [20]:
#HUMAN FRIENDLY DATA PRESENTATION

display_df = df_final.rename(columns={
    "customer_id": "Customer ID",
    "purchase_amount_usd": "Purchase Amount ($)",
    "age_group": "Age Group",
    "spend_tier": "Spend Tier",
    "customer_value_score": "Value Score",
    "is_subscribed": "Subscribed",
    "frequency_of_purchases": "Purchase Frequency"
})
display_df.head()

,Customer ID,age,Age Group,gender,category,Purchase Amount ($),Spend Tier,location,size,color,...,payment_method,preferred_payment_method,shipping_type,discount_applied,discount_used,promo_code_used,promo_used,previous_purchases,Value Score,Purchase Frequency
0,1,55,46-55,Male,Clothing,53.0,Medium,Kentucky,L,Gray,...,Credit Card,Venmo,Express,Yes,1,Yes,1,14,70.0,Fortnightly
1,2,19,18-25,Male,Clothing,64.0,High,Maine,L,Maroon,...,Bank Transfer,Cash,Express,Yes,1,Yes,1,2,51.5,Fortnightly
2,3,50,46-55,Male,Clothing,73.0,High,Massachusetts,S,Maroon,...,Cash,Credit Card,Free Shipping,Yes,1,Yes,1,23,98.0,Weekly
3,4,21,18-25,Male,Footwear,90.0,High,Rhode Island,M,Maroon,...,Paypal,Paypal,Next Day Air,Yes,1,Yes,1,49,160.5,Weekly
4,5,45,36-45,Male,Clothing,49.0,Medium,Oregon,M,Turquoise,...,Cash,Paypal,Free Shipping,Yes,1,Yes,1,31,100.0,Annually


In [25]:
!pip install pymysql sqlalchemy

In [29]:
from sqlalchemy import create_engine

# XAMPP default: user=root, password="" (empty), port=3306
engine = create_engine("mysql+pymysql://root:@localhost:3306/customerssales")

df.to_sql(
    name="customer_shopping",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=1000
)

print("Uploaded", len(df), "rows to customerssales.customer_shopping")


Uploaded 3900 rows to customerssales.customer_shopping
